We check the decomposition of a $\mathrm{CNOT}_{12}$ with qubit $1$ as control and $2$ as target followed by a $\mathrm{CPHASE}_{13}(\theta)$ acting on qubits $1$ and $3$. This is a particular case of the decomposition described in Appendix B of M. Schumann et al "Bridging wire and gate cutting with ZX-calculus" (2025)

In [1]:
import numpy as np
from util import return_ptm
import scipy

In [2]:
def apply_mcphase_theta(op: np.ndarray, params: dict):
    n = int(np.log2(op.shape[1]))
    if op.shape[1] != 2**n:
        raise ValueError("Matrix shape error: the matrix dimension must be a power of 2")
    theta = params["theta"]
    mcphase = np.identity(2**n, dtype=complex)
    mcphase[-1, -1] = np.exp(1j*theta)
    return mcphase@op@mcphase.conj().T 

def apply_cnot12_cphase13(op: np.ndarray, params: dict):
    n = int(np.log2(op.shape[1]))
    if op.shape[1] != 8:
        raise ValueError("Matrix shape error: the matrix dimension must be 8x8")
    theta = params["theta"]
    cnot = np.identity(4, dtype=complex)
    cnot[2:4, 2:4] = np.array([[0, 1], [1, 0]], dtype=complex)
    swap = np.identity(4, dtype=complex)
    swap[1:3, 1:3] = np.array([[0, 1], [1, 0]], dtype=complex)
    cphase = np.identity(2**2, dtype=complex)
    cphase[-1, -1] = np.exp(1j*theta)
    cnot12 = np.kron(cnot, np.identity(2))
    cphase12 = np.kron(cphase, np.identity(2))
    swap23 = np.kron(np.identity(2), swap)
    cphase13 = swap23@cphase12@swap23
    u = cphase13@cnot12
    return u@op@u.conj().T 

def apply_bottom_y_term(op: np.ndarray, params: dict):
    n = int(np.log2(op.shape[1]))
    if op.shape[1] != 2**2:
        raise ValueError("Matrix shape error: the matrix dimension must be a 4x4")
    ancilla_state = 1/np.sqrt(2)*np.array([[1], [1]])
    rho_plus_i = 1/2*np.array([[1, -1j], [1j, 1]])
    rho_minus_i = 1/2*np.array([[1, 1j], [-1j, 1]])
    rho_ancilla_state = ancilla_state@ancilla_state.conj().T
    op_with_ancilla = np.kron(rho_ancilla_state, op)
    op_with_ancilla_new = apply_cnot12_cphase13(op_with_ancilla, {"theta": params["theta"]})
    proj_plus_i = np.kron(rho_plus_i, np.identity(2**n))
    proj_minus_i = np.kron(rho_minus_i, np.identity(2**n))
    op_proj_plus_i = proj_plus_i@op_with_ancilla_new@proj_plus_i
    op_proj_minus_i = proj_minus_i@op_with_ancilla_new@proj_minus_i
    # The following takes the partial trace with respect to the first subsystem
    op_plus_i_trace = np.trace(op_proj_plus_i.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    op_minus_i_trace = np.trace(op_proj_minus_i.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    # op_trace = np.trace(op_with_ancilla_mcz.reshape(2**n , 2, 2**n, 2), axis1=1, axis2=3) # only for testing
    if params["result"] == 0:
        return op_plus_i_trace
    elif params["result"] == 1:
        return op_minus_i_trace

def apply_bottom_x_term(op: np.ndarray, params: dict):
    n = int(np.log2(op.shape[1]))
    if op.shape[1] != 2**2:
        raise ValueError("Matrix shape error: the matrix dimension must be a 4x4")
    ancilla_state = 1/np.sqrt(2)*np.array([[1], [1]])
    rho_plus = 1/2*np.array([[1, 1], [1, 1]])
    rho_minus = 1/2*np.array([[1, -1], [-1, 1]])
    rho_ancilla_state = ancilla_state@ancilla_state.conj().T
    op_with_ancilla = np.kron(rho_ancilla_state, op)
    op_with_ancilla_new = apply_cnot12_cphase13(op_with_ancilla, {"theta": params["theta"]})
    proj_plus = np.kron(rho_plus, np.identity(2**n))
    proj_minus = np.kron(rho_minus, np.identity(2**n))
    op_proj_plus = proj_plus@op_with_ancilla_new@proj_plus
    op_proj_minus = proj_minus@op_with_ancilla_new@proj_minus
    # The following takes the partial trace with respect to the first subsystem
    op_plus_trace = np.trace(op_proj_plus.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    op_minus_trace = np.trace(op_proj_minus.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    # op_trace = np.trace(op_with_ancilla_mcz.reshape(2**n , 2, 2**n, 2), axis1=1, axis2=3) # only for testing      
    return op_plus_trace - op_minus_trace

def apply_bottom_z_term(op: np.ndarray, params: dict):
    n = int(np.log2(op.shape[1]))
    if op.shape[1] != 2**2:
        raise ValueError("Matrix shape error: the matrix dimension must be a 4x4")
    ancilla_state = 1/np.sqrt(2)*np.array([[1], [1]])
    rho_plus = np.array([[1, 0], [0, 0]])
    rho_minus = np.array([[0, 0], [0, 1]])
    rho_ancilla_state = ancilla_state@ancilla_state.conj().T
    op_with_ancilla = np.kron(rho_ancilla_state, op)
    op_with_ancilla_new = apply_cnot12_cphase13(op_with_ancilla, {"theta": params["theta"]})
    proj_plus = np.kron(rho_plus, np.identity(2**n))
    proj_minus = np.kron(rho_minus, np.identity(2**n))
    op_proj_plus = proj_plus@op_with_ancilla_new@proj_plus
    op_proj_minus = proj_minus@op_with_ancilla_new@proj_minus
    # The following takes the partial trace with respect to the first subsystem
    op_plus_trace = np.trace(op_proj_plus.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    op_minus_trace = np.trace(op_proj_minus.reshape(2, 2**n , 2, 2**n), axis1=0, axis2=2)
    # op_trace = np.trace(op_with_ancilla_mcz.reshape(2**n , 2, 2**n, 2), axis1=1, axis2=3) # only for testing      
    return op_plus_trace - op_minus_trace

def apply_rz_theta(op: np.ndarray, params: dict):
    if op.shape[1] != 2:
        raise ValueError("Matrix shape error: the matrix should be 2x2")
    pauli_z = np.array([[1, 0], [0, -1]])
    theta = params["theta"]
    rz = scipy.linalg.expm(-1j*theta*pauli_z/2)
    return rz@op@rz.conj().T

def apply_rx_theta(op: np.ndarray, params: dict):
    if op.shape[1] != 2:
        raise ValueError("Matrix shape error: the matrix should be 2x2")
    pauli_x = np.array([[0, 1], [1, 0]])
    theta = params["theta"]
    rx = scipy.linalg.expm(-1j*theta*pauli_x/2)
    return rx@op@rx.conj().T

def apply_measure_and_prepare_z(op: np.ndarray, params=None):
    if op.shape[1] != 2:
        raise ValueError("Matrix shape error: the matrix should be 2x2")
    rho_0 = np.array([[1, 0], [0, 0]])
    rho_1 = np.array([[0, 0], [0, 1]])
    return rho_0*np.trace(rho_0@op) - rho_1*np.trace(rho_1@op)
    

In [3]:
num_qubits = 3
m = 1
m_prime = num_qubits - m
theta = np.pi/3

In [4]:
target_ptm = return_ptm(apply_cnot12_cphase13, num_qubits, params={"theta": theta})

In [5]:
def apply_cnot(op: np.ndarray, params: dict=None):
    if op.shape[1] != 4:
        raise ValueError("Matrix shape error: the matrix should be 4x4")
    cnot = np.block([[np.identity(2), np.zeros([2, 2])], [np.zeros([2, 2]), np.array([[0, 1], [1, 0]])]])
    return cnot@op@cnot.conj().T

def apply_swap(op: np.ndarray, params: dict=None):
    if op.shape[1] != 4:
        raise ValueError("Matrix shape error: the matrix should be 4x4")
    swap = np.identity(4)
    swap[1:3, 1:3] = np.array([[0, 1], [1, 0]])
    return swap@op@swap.conj().T


Now we obtain the PTMs for each term in the decomposition
1. $Y$-term (requires classical communication)

In [6]:
deco_dict = {}
deco_dict["1"] = {}
deco_dict["1"]["q"] = 1
deco_dict["1"]["ptm"] = np.kron(return_ptm(apply_rz_theta, m, params={"theta": np.pi/2}), return_ptm(apply_bottom_y_term, m_prime, params={"theta": theta, "result": 0})) 
deco_dict["1"]["ptm"] += np.kron(return_ptm(apply_rz_theta, m, params={"theta": -np.pi/2}), return_ptm(apply_bottom_y_term, m_prime, params={"theta": theta, "result": 1}))

2. X-Term $0$

In [7]:
deco_dict["2"] = {}
deco_dict["2"]["q"] = 1/2
deco_dict["2"]["ptm"] = np.kron(return_ptm(apply_rz_theta, m, params={"theta": 0.0}), return_ptm(apply_bottom_x_term, m_prime, params={"theta": theta}))

3. X-term $1$

In [8]:
deco_dict["3"] = {} 
deco_dict["3"]["q"] = -1/2
deco_dict["3"]["ptm"] = np.kron(return_ptm(apply_rz_theta, m, params={"theta": np.pi}), return_ptm(apply_bottom_x_term, m_prime, params={"theta": theta}))

4. Z-term

In [9]:
deco_dict["4"] = {}
deco_dict["4"]["q"] = 1
deco_dict["4"]["ptm"] = np.kron(return_ptm(apply_measure_and_prepare_z, 1, params=None), return_ptm(apply_bottom_z_term, m_prime, params={"theta": theta}))


In [10]:
cut = sum([deco_dict[key]["q"]*deco_dict[key]["ptm"] for key in deco_dict.keys()])

In [11]:
np.max(np.abs(target_ptm - cut))

np.float64(2.220893329053004e-16)

which shows that the decomposition is correct!